In [5]:
import altair as alt
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsRegressor

# Simplify working with large datasets in Altair
alt.data_transformers.enable('vegafusion')

# Output dataframes instead of arrays
set_config(transform_output="pandas")

In [6]:
players_raw = pd.read_csv("players.csv")
players_raw

,experience,subscribe,hashedEmail,played_hours,name,gender,age,individualId,organizationName
0,Pro,True,f6daba428a5e19a3d47574858c13550499be23603422e6...,30.3,Morgan,Male,9,NaN,NaN
1,Veteran,True,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa9397...,3.8,Christian,Male,17,NaN,NaN
2,Veteran,False,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3...,0.0,Blake,Male,17,NaN,NaN
3,Amateur,True,23fe711e0e3b77f1da7aa221ab1192afe21648d47d2b4f...,0.7,Flora,Female,21,NaN,NaN
4,Regular,True,7dc01f10bf20671ecfccdac23812b1b415acd42c2147cb...,0.1,Kylie,Male,21,NaN,NaN
...,...,...,...,...,...,...,...,...,...
191,Amateur,True,b6e9e593b9ec51c5e335457341c324c34a2239531e1890...,0.0,Bailey,Female,17,NaN,NaN
192,Veteran,False,71453e425f07d10da4fa2b349c83e73ccdf0fb3312f778...,0.3,Pascal,Male,22,NaN,NaN
193,Amateur,False,d572f391d452b76ea2d7e5e53a3d38bfd7499c7399db29...,0.0,Dylan,Prefer not to say,17,NaN,NaN
194,Amateur,False,f19e136ddde68f365afc860c725ccff54307dedd13968e...,2.3,Harlow,Male,17,NaN,NaN


In [7]:
sessions_raw = pd.read_csv("sessions.csv")
sessions_raw

,hashedEmail,start_time,end_time,original_start_time,original_end_time
0,bfce39c89d6549f2bb94d8064d3ce69dc3d7e72b38f431...,30/06/2024 18:12,30/06/2024 18:24,1.719770e+12,1.719770e+12
1,36d9cbb4c6bc0c1a6911436d2da0d09ec625e43e6552f5...,17/06/2024 23:33,17/06/2024 23:46,1.718670e+12,1.718670e+12
2,f8f5477f5a2e53616ae37421b1c660b971192bd8ff77e3...,25/07/2024 17:34,25/07/2024 17:57,1.721930e+12,1.721930e+12
3,bfce39c89d6549f2bb94d8064d3ce69dc3d7e72b38f431...,25/07/2024 03:22,25/07/2024 03:58,1.721880e+12,1.721880e+12
4,36d9cbb4c6bc0c1a6911436d2da0d09ec625e43e6552f5...,25/05/2024 16:01,25/05/2024 16:12,1.716650e+12,1.716650e+12
...,...,...,...,...,...
1530,36d9cbb4c6bc0c1a6911436d2da0d09ec625e43e6552f5...,10/05/2024 23:01,10/05/2024 23:07,1.715380e+12,1.715380e+12
1531,7a4686586d290c67179275c7c3dfb4ea02f4d317d9ee0e...,01/07/2024 04:08,01/07/2024 04:19,1.719810e+12,1.719810e+12
1532,fd6563a4e0f6f4273580e5fedbd8dda64990447aea5a33...,28/07/2024 15:36,28/07/2024 15:57,1.722180e+12,1.722180e+12
1533,fd6563a4e0f6f4273580e5fedbd8dda64990447aea5a33...,25/07/2024 06:15,25/07/2024 06:22,1.721890e+12,1.721890e+12


### (1) Data Description
#### players
This file contains information about every individual player each with a unique hashed email. There are nine variables (columns) in total, namely *experience*, *subscribe*, *hashedEmail*, *played_hours*, *name*, *gender*, *age*, *IndividualId* and *OrganisationName*. There are 196 players are on this record. The variable type and meaning are listed as follows:
+ Experience: A categorical variable which classifies players into Amateur, Regular, Veteran and Pro.
+ Subscribe: A categorical variable which provides the player has subscribed to the game-related newsletter or not (True/False).
+ Hashed Email: A unique identifier (key) to track each invidual player across the data sets.
+ Played Hours: Numerical continuous variable which demonstrates the total playing hours of the player.
+ Name: The registered (character) name of the player on the server.
+ Gender: A categorical variable which indicates the claimed gender of the player.
+ Age: A continuous numerical variable which lists the claimed age of the player.
+ IndividualId: The individual ID of the player. It is None value in this data set.
+ Organisation: The organisation that the player represent. It is None value in this data set.

The None values in the data shall be dropped as it does not contribute to the analysis and save computational resources. Nonetheless, one should me reminded that the gender and age are said to be "claimed" values as the server requires No verification process. For instance, there is a player who aged 91 by playing 0.2 hours only. The only True (same as reality) values of this data set is the play hours, subscription and experience, however this also comes with fallacies. The player might have different accounts incurring the situation of limited play hours but being skillful for the game. We shall identify these discrepencies and modify the data set as it would reflecting the true case. 

#### sessions
This file contains information about every individual game session played by the players. It consists of 5 columns (variables) in total with 1535 rows. Namely *hashedEmail*, *start_time*, *end_time*, *original_start_time* and *original_end_time*. The variable type and meaning of each variable are listed as following:
+ Hashed Email: A unique identifier (key) to track each invidual player across the data sets.
+ start time: Start time of the game session in human-readable time.
+ end time: End time of the game session in human-readable time.
+ Original Start Time: Start time of the game session in Unix Epoch timestamp in milliseconds.
+ Original End Time: End time of the game session in Unix Epoch timestamp in milliseconds.

The data consists of every individual gaming session, in other words, it could not be directly merged to the players data set. Moreover, single record are not useful for data analysist as it corresponds to discrete record of every playing session. This data set shall be wrangled alternatively and merge in to the original session. For instance, by combining the gaming session of individual players, calculate the total number of gaming sessions the player contributes, and by then calculate the average gaming time everytime the player starts a session.By so we could reveal the gaming habit of the player. 

### (2) Question  
#### Question 2: Which "kinds" of players are most likely to contribute a large amount of data?  

The goal of this project is to identify the demographic and behavioral characteristics of **each player group** (Amateur, Regular, Veteran, Pro) that are associated with contributing large amounts of data to the research server.  

Here, **“large amount of data”** will be defined in terms of:  
- **Total play hours** (from `players.csv`)  
- **Number of sessions and average session length** (aggregated from `sessions.csv`)  

The **response variable** will be player activity (e.g., cumulative session time or total hours played).  

The **explanatory variables** will include:  
- **Experience level** (Amateur, Regular, Veteran, Pro)  
- **Subscription status** (True/False)  
- **Demographics** (age, gender)  
- **Gaming habits** (average session length, frequency of sessions)  

Rather than focusing on a single predictor, the analysis will examine **combinations of demographic and behavioral features within each experience group**. For example:  
- Amateurs who average ≥15 minutes per session may represent ~50% of their group contributing substantial data.  
- Pros who average ≥2 hour per session may represent a majority of their group contributing substantial data.  

To achieve this, I will:  
1. **Wrangle `sessions.csv`** to compute per-player aggregates (total sessions, average session duration, cumulative session time).  
2. **Merge with `players.csv`** using `hashedEmail` to link demographics and subscription status.  
3. **Profile each experience group** by summarizing distributions of activity and demographics.  
4. **Identify thresholds/ranges** (e.g., median or top quartile of session length) that distinguish players who contribute disproportionately large amounts of data.  

This approach will allow stakeholders to see not only which player groups are most active overall, but also which **subsets within each group** (defined by ranges of demographics and gaming habits) are most likely to provide meaningful contributions.  

### (3) Exploratory Data Analysis and Visualization  

To explore which characteristics are associated with newsletter subscription, we examined how player experience, demographics, and gaming activity relate to subscription status.  

The bar chart shows that **subscription rates vary only slightly across experience levels**, which suggests that experience alone is not a strong predictor of subscription. Overall, the ratio of subscribers is relatively consistent across groups, meaning that experience level by itself does not explain subscription behavior.  

The boxplot of average session length indicates that **subscribed players generally have longer sessions**, with a higher median and wider spread compared to non-subscribers. The scatterplots of total session time show that **subscribed players tend to accumulate more hours overall**, with several individuals contributing exceptionally high totals.  

Demographic variables also provide insight: the scatterplot of age versus total session time shows that younger subscribed players are more likely to contribute large amounts of data, while non-subscribers cluster at lower totals. The bar chart of gender reveals variation in subscription rates across categories, with some groups showing very high subscription rates and others much lower.  

Overall, these exploratory results suggest that **relative activity (session length and total time) is most closely linked to subscription, while demographics such as age and gender add secondary but meaningful variation**. This motivates a classification approach that combines both behavioral and demographic predictors to better understand subscription behavior.

In [8]:
# Drop irrelevant columns
players = players_raw.drop(columns=["IndividualId", "OrganisationName"], errors="ignore")

# Standardize column names
players.columns = players.columns.str.lower()
sessions = sessions_raw.copy()
sessions.columns = sessions.columns.str.lower()

# Clean players: drop missing key fields
players = players.dropna(subset=["hashedemail", "subscribe", "experience"])
players["subscribe"] = players["subscribe"].astype(str).str.lower().map({"true": True, "false": False})

# Clean sessions: parse datetime safely
sessions = sessions.dropna(subset=["hashedemail", "start_time", "end_time"])
sessions["start_time"] = pd.to_datetime(sessions["start_time"], format="%d/%m/%Y %H:%M", errors="coerce")
sessions["end_time"] = pd.to_datetime(sessions["end_time"], format="%d/%m/%Y %H:%M", errors="coerce")
sessions = sessions.dropna(subset=["start_time", "end_time"])

# Compute session duration in hours
sessions["duration"] = sessions["end_time"] - sessions["start_time"]
sessions["duration_hours"] = sessions["duration"].apply(lambda x: x.total_seconds() / 3600 if pd.notnull(x) else np.nan)
sessions = sessions[sessions["duration_hours"] >= 0]

# Aggregate session features per player
agg = (
    sessions.groupby("hashedemail")
    .agg(
        total_sessions=("hashedemail", "count"),
        avg_session_length=("duration_hours", "mean"),
        total_session_time=("duration_hours", "sum")
    )
    .reset_index()
)

# Merge with players
df = pd.merge(players, agg, on="hashedemail", how="left")
df[["total_sessions", "avg_session_length", "total_session_time"]] = df[
    ["total_sessions", "avg_session_length", "total_session_time"]
].fillna(0)

# Cap outliers at 99th percentile
for col in ["avg_session_length", "total_session_time", "total_sessions"]:
    cap = df[col].quantile(0.99)
    df[col] = np.where(df[col] > cap, cap, df[col])

# Convert experience to category
df["experience"] = df["experience"].astype("category")

# Plot 1: Subscription rate by experience
sub_rate = df.groupby("experience")["subscribe"].mean().reset_index()
sub_ex = alt.Chart(sub_rate).mark_bar().encode(
    x=alt.X("experience:N", title="Experience Level"),
    y=alt.Y("subscribe:Q", title="Subscription Rate"),
    tooltip=["experience", alt.Tooltip("subscribe:Q", format=".2f")]
).properties(title="Subscription Rate by Experience Level")

# Plot 2: Average session length by subscription
sub_avg = alt.Chart(df).mark_boxplot().encode(
    x=alt.X("subscribe:N", title="Subscribed"),
    y=alt.Y("avg_session_length:Q", title="Average Session Length (hours)"),
    color="subscribe:N"
).properties(title="Average Session Length by Subscription Status")

# Plot 3: Total session time by subscription
sub_tot = alt.Chart(df).mark_boxplot().encode(
    x=alt.X("subscribe:N", title="Subscribed"),
    y=alt.Y("total_session_time:Q", title="Total Session Time (hours)"),
    color="subscribe:N"
).properties(title="Total Session Time by Subscription Status")

# Plot 4: Age vs. Subscription status
chart_age = alt.Chart(df).mark_circle(size=60, opacity=0.6).encode(
    x=alt.X("age:Q", title="Age (years)"),
    y=alt.Y("total_session_time:Q", title="Total Session Time (hours)"),
    color=alt.Color("subscribe:N", title="Subscribed", scale=alt.Scale(domain=[True, False], range=["orange", "steelblue"])),
    tooltip=["age", "gender", "experience", "total_session_time", "subscribe"]
).properties(title="Total Session Time vs. Age, colored by Subscription Status")

# Plot 5: Gender vs. Subscription rate
sub_rate_gender = df.groupby("gender")["subscribe"].mean().reset_index()
chart_gender = alt.Chart(sub_rate_gender).mark_bar().encode(
    x=alt.X("gender:N", title="Gender"),
    y=alt.Y("subscribe:Q", title="Subscription Rate"),
    tooltip=["gender", alt.Tooltip("subscribe:Q", format=".2f")]
).properties(title="Subscription Rate by Gender")

sub_ex | (sub_avg & sub_tot) | (chart_age & chart_gender)



/tmp/ipykernel_205/3244493048.py:50: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sub_rate = df.groupby("experience")["subscribe"].mean().reset_index()


alt.HConcatChart(...)

### (4) Methods and Plan  

To predict whether a player subscribes to the newsletter, I will use **K-nearest neighbors (KNN) classification** as the primary method. KNN is appropriate because the response variable is binary (subscribe: True/False), and the algorithm can capture similarities in player activity and demographics without requiring strong distributional assumptions. Players with similar gaming habits (e.g., session length, total time) and demographic profiles (age, gender, experience) are expected to have similar subscription outcomes.  

In addition, I will use **linear regression** to explore continuous outcomes such as **total play hours** or **cumulative session time**. This allows me to quantify how explanatory variables (experience, subscription status, demographics, average session length, number of sessions) contribute to overall activity levels. Linear regression is appropriate here because the response variable is continuous, and the model provides interpretable coefficients that show the direction and strength of each predictor’s effect.  

The exploratory analysis highlighted potential issues:  
- **Outliers** in total session time (players with exceptionally high hours) may skew results.  
- **Small gender categories** may lead to unstable estimates.  
These will be addressed by dropping extreme values and carefully interpreting coefficients for underrepresented groups.  

To evaluate the models, I will split the dataset into training and test sets (80/20 split) and use **cross-validation** on the training set to select the best hyperparameters (e.g., the value of *k* for KNN). Model performance will be compared using **accuracy and misclassification rate** for classification, and **mean squared error (MSE)** for regression, consistent with course practices. Numeric features such as age, session length, and total time will be standardized, and categorical variables (e.g., gender, experience level) will be encoded.  

This plan ensures that the analysis directly tests whether **active gaming habits and demographic characteristics are predictive of subscription**, while also quantifying how these features contribute to overall activity levels. Both methods are fully aligned with the course material and provide complementary insights: KNN classification for subscription prediction, and linear regression for continuous measures of player activity. By so, it helps the research team identify which players are most likely to generate meaningful data